# 02 - Preprocessing & Train/Validation/Test Split

Goals:
1. Load raw data
2. Run leakage checks
3. Stratified split into train / validation / test (preserve fraud rate)
4. Save splits to `data/processed/`


In [ ]:
import sys
sys.path.append("..")

from src.data import load_data, check_leakage, stratified_split
import pandas as pd

In [ ]:
df = load_data("../data/raw/credit_card_fraud_10k.csv")
df.head()

In [ ]:
# Leakage check
check_leakage(df, target="is_fraud")
print("No leakage detected.")

In [ ]:
# Stratified split: 60% train / 20% val / 20% test
train, val, test = stratified_split(df, target="is_fraud", test_size=0.2, val_size=0.25, random_state=42)

print("Train shape:", train.shape, "| Fraud rate: {:.2f}%".format(100 * train["is_fraud"].mean()))
print("Val   shape:", val.shape, "   | Fraud rate: {:.2f}%".format(100 * val["is_fraud"].mean()))
print("Test  shape:", test.shape, "  | Fraud rate: {:.2f}%".format(100 * test["is_fraud"].mean()))

In [ ]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

train.to_csv(processed_dir / "train.csv", index=False)
val.to_csv(processed_dir / "val.csv", index=False)
test.to_csv(processed_dir / "test.csv", index=False)

print("Saved splits to", processed_dir)

## Why stratified split?
Fraud is rare. A random split could put all (or no) fraud cases in test. Stratification preserves the fraud rate in every split so PR-AUC and F1-macro are evaluated fairly.